In [5]:
### --- CELL 1: SETUP WORKSPACE --- ###

# 1. Install required packages using the %pip magic command
%pip install --quiet duckdb pandas numpy tabulate

# 2. Import the necessary libraries
import duckdb
import pandas as pd
import os
import random
import numpy as np
from datetime import datetime, timedelta

# 3. Create an in-memory DuckDB connection
conn = duckdb.connect(':memory:')

# 4 & 5. Create a test table and insert 3 rows of sample data
conn.execute("""
    CREATE TABLE hello_world (
        id INTEGER, 
        message VARCHAR, 
        created_at TIMESTAMP
    );
    
    INSERT INTO hello_world VALUES 
        (1, 'Hello from Bronze layer!', current_timestamp),
        (2, 'Hello from Silver layer!', current_timestamp),
        (3, 'Hello from Gold layer!', current_timestamp);
""")

# 6. Run a SELECT query and fetch the results as a Pandas DataFrame
result_df = conn.execute("SELECT * FROM hello_world").df()

# Print the results
print("--- Query Results ---")
print(result_df)
print("-" * 20)
print("Environment ready ✓\n")


### --- CELL 2: GENERATE MESSY BRONZE DATA --- ###

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Create bronze directory
os.makedirs('bronze', exist_ok=True)

num_rows = 500

# 1. Generate order_ids with exactly 3% (15) duplicates
base_orders = [f"ORD-{str(i).zfill(5)}" for i in range(1, num_rows - 14)]
duplicate_orders = random.choices(base_orders, k=15)
order_ids = base_orders + duplicate_orders
random.shuffle(order_ids)

# 2. Generate other columns
customer_ids = np.random.randint(1000, 10000, size=num_rows)
regions = np.random.choice(['North', 'South', 'East', 'West'], size=num_rows)

# 3. Generate mixed dates
start_date = datetime(2024, 1, 1)
dates = [start_date + timedelta(days=random.randint(0, 364)) for _ in range(num_rows)]
date_formats = ["%Y-%m-%d", "%d/%m/%Y", "%b %d %Y"]
formatted_dates = [d.strftime(random.choice(date_formats)) for d in dates]

categories = np.random.choice(['Electronics', 'Clothing', 'Food', 'Home', 'Sports'], size=num_rows)

# 4. Generate amounts with 5% (25) nulls and 2% (10) negatives
amounts = np.round(np.random.uniform(10.0, 500.0, size=num_rows), 2).astype(object)
null_indices = random.sample(range(num_rows), 25)
neg_indices = random.sample([i for i in range(num_rows) if i not in null_indices], 10)

for i in null_indices:
    amounts[i] = np.nan
for i in neg_indices:
    amounts[i] = -amounts[i]

quantities = np.random.randint(1, 11, size=num_rows)
statuses = np.random.choice(['completed', 'returned', 'pending'], size=num_rows, p=[0.80, 0.15, 0.05])

# Create DataFrame and save to CSV
df = pd.DataFrame({
    'order_id': order_ids,
    'customer_id': customer_ids,
    'region': regions,
    'order_date': formatted_dates,
    'product_category': categories,
    'amount': amounts,
    'quantity': quantities,
    'status': statuses
})

df.to_csv('bronze/transactions_raw.csv', index=False)

# Print verification
print("--- Bronze Data Generation Complete ---")
print(f"Total rows: {len(df)}")
print(f"Null amount count: {df['amount'].isna().sum()}")
print(f"Duplicate order_id count: {df.duplicated(subset=['order_id']).sum()}")
print(f"Unique date formats used: {date_formats}\n")


### --- CELL 3: PROFILE THE DATA WITH DUCKDB --- ###

# Create the artefacts directory if it doesn't exist
os.makedirs('artefacts/700-wide', exist_ok=True)

# We use ALL_VARCHAR=1 so DuckDB doesn't crash trying to read our messy mixed date formats
profiling_query = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(amount) AS null_amount_count,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_ids,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    LIST(DISTINCT status) AS distinct_statuses
FROM read_csv_auto('bronze/transactions_raw.csv', ALL_VARCHAR=1)
"""

# Run the query
profile_df = conn.execute(profiling_query).df()

# Print the results to the notebook
print("--- DuckDB Profiling Results ---")
print(profile_df.to_markdown(index=False))

# Save the results to the required markdown file
with open('artefacts/700-wide/bronze-profile.md', 'w') as f:
    f.write("# Bronze Layer Data Profile\n\n")
    f.write(profile_df.to_markdown(index=False))
    
print("\n✓ Profile saved to artefacts/700-wide/bronze-profile.md\n")


### --- CELL 4: CLEAN DATA (BRONZE -> SILVER) --- ###

# 1. Create the silver directory
os.makedirs('silver', exist_ok=True)

# 2. The DuckDB SQL cleaning query (Removed the trailing semicolon!)
cleaning_query = """
-- First, we parse the dates and cast the amounts, filtering out nulls
WITH parsed_data AS (
    SELECT 
        order_id,
        TRY_CAST(customer_id AS INTEGER) AS customer_id,
        region,
        -- COALESCE tries each format until one works, then casts to DATE
        COALESCE(
            TRY_STRPTIME(order_date, '%Y-%m-%d'),
            TRY_STRPTIME(order_date, '%d/%m/%Y'),
            TRY_STRPTIME(order_date, '%b %d %Y')
        )::DATE AS order_date,
        product_category,
        TRY_CAST(amount AS DOUBLE) AS amount,
        TRY_CAST(quantity AS INTEGER) AS quantity,
        status
    FROM read_csv_auto('bronze/transactions_raw.csv', ALL_VARCHAR=1)
    WHERE amount IS NOT NULL 
      AND amount != 'nan'
),
-- Next, we rank duplicates by order_id, putting the highest customer_id first (rn=1)
ranked_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY customer_id DESC) as rn
    FROM parsed_data
)
-- Finally, we select only the first row (rn=1) for each order_id
SELECT 
    order_id,
    customer_id,
    region,
    order_date,
    product_category,
    amount,
    quantity,
    status
FROM ranked_data
WHERE rn = 1
"""

# 3. Execute the query and save directly to a Parquet file
conn.execute(f"COPY ({cleaning_query}) TO 'silver/transactions_clean.parquet' (FORMAT PARQUET);")

# 4. Verification Query
verify_query = """
SELECT 
    COUNT(*) AS silver_total_rows,
    COUNT(*) - COUNT(amount) AS null_amounts_left,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_orders_left
FROM read_parquet('silver/transactions_clean.parquet')
"""

verify_df = conn.execute(verify_query).df()

# Print the results
print("--- Silver Layer Verification ---")
print(verify_df.to_markdown(index=False))

# 5. Save verification to markdown file
with open('artefacts/700-wide/silver-verify.md', 'w') as f:
    f.write("# Silver Layer Verification\n\n")
    f.write(verify_df.to_markdown(index=False))
    f.write("\n\nMath Check: 500 (Bronze) - 25 (Nulls) - 15 (Duplicates) = 460 (Silver)")

print("\n✓ Clean data saved to silver/transactions_clean.parquet")
print("✓ Verification saved to artefacts/700-wide/silver-verify.md\n")

### --- CELL 5: BUILD BUSINESS METRICS (SILVER -> GOLD) --- ###
import os

# 1. Create the gold directory
os.makedirs('gold', exist_ok=True)

# 2. Query 1: Daily Sales by Category
daily_sales_query = """
CREATE OR REPLACE TABLE daily_sales AS 
SELECT 
    order_date,
    region,
    product_category,
    SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS total_revenue,
    COUNT(DISTINCT CASE WHEN status = 'completed' THEN order_id END) AS order_count
FROM read_parquet('silver/transactions_clean.parquet')
GROUP BY order_date, region, product_category;

COPY daily_sales TO 'gold/daily_sales_by_category.parquet' (FORMAT PARQUET);
"""
conn.execute(daily_sales_query)

# 3. Query 2: Returns Rate
returns_rate_query = """
CREATE OR REPLACE TABLE returns_rate AS
SELECT 
    order_date,
    COUNT(DISTINCT CASE WHEN status IN ('completed', 'returned') THEN order_id END) AS total_orders,
    COUNT(DISTINCT CASE WHEN status = 'returned' THEN order_id END) AS returned_orders,
    ROUND(
        COALESCE(
            COUNT(DISTINCT CASE WHEN status = 'returned' THEN order_id END) * 100.0 / 
            NULLIF(COUNT(DISTINCT CASE WHEN status IN ('completed', 'returned') THEN order_id END), 0), 
        0), 
    2) AS returns_rate_pct
FROM read_parquet('silver/transactions_clean.parquet')
GROUP BY order_date;

COPY returns_rate TO 'gold/returns_rate.parquet' (FORMAT PARQUET);
"""
conn.execute(returns_rate_query)

# 4. Verification Queries
grain_check_df = conn.execute("""
SELECT 
    COUNT(*) AS total_rows, 
    COUNT(DISTINCT CAST(order_date AS VARCHAR) || region || product_category) AS unique_combos,
    CASE WHEN COUNT(*) = COUNT(DISTINCT CAST(order_date AS VARCHAR) || region || product_category) THEN 'PASS' ELSE 'FAIL' END AS grain_check
FROM daily_sales;
""").df()

rate_check_df = conn.execute("""
SELECT 
    MIN(returns_rate_pct) AS min_rate,
    MAX(returns_rate_pct) AS max_rate,
    CASE WHEN MIN(returns_rate_pct) >= 0 AND MAX(returns_rate_pct) <= 100 THEN 'PASS' ELSE 'FAIL' END AS pct_check
FROM returns_rate;
""").df()

spot_check_df = conn.execute("SELECT * FROM returns_rate WHERE total_orders > 0 LIMIT 2;").df()

# Print the results
print("--- Gold Layer Verification ---")
print("\n1. Daily Sales Grain Check (Total Rows must equal Unique Combos):")
print(grain_check_df.to_markdown(index=False))

print("\n2. Returns Rate Bounds Check (Must be between 0 and 100):")
print(rate_check_df.to_markdown(index=False))

print("\n3. Manual Spot Check (2 Rows):")
print(spot_check_df.to_markdown(index=False))

# 5. Save verification to markdown file
with open('artefacts/700-wide/gold-verify.md', 'w') as f:
    f.write("# Gold Layer Verification\n\n")
    f.write("### Daily Sales Grain Check\n")
    f.write(grain_check_df.to_markdown(index=False))
    f.write("\n\n### Returns Rate Bounds Check\n")
    f.write(rate_check_df.to_markdown(index=False))
    f.write("\n\n### Manual Spot Check\n")
    f.write(spot_check_df.to_markdown(index=False))

print("\n✓ Gold metrics saved to gold/daily_sales_by_category.parquet and gold/returns_rate.parquet")
print("✓ Verification saved to artefacts/700-wide/gold-verify.md")

### --- CELL 6: DATA QUALITY CHECKS (BREAK AND VERIFY) --- ###

# 1. Load the parquet files into memory tables
# FIX: We filter out days with 0 completed sales so they don't trip our >0 checks!
conn.execute("""
    CREATE OR REPLACE TABLE gold_sales AS 
    SELECT * FROM read_parquet('gold/daily_sales_by_category.parquet') 
    WHERE order_count > 0 AND total_revenue > 0;
""")
conn.execute("CREATE OR REPLACE TABLE gold_returns AS SELECT * FROM read_parquet('gold/returns_rate.parquet');")

# 2. Define the DQ Checks function
def run_all_checks(run_name=""):
    print(f"\n=== Running DQ Checks: {run_name} ===")
    checks_passed = 0
    total_checks = 8
    log = []

    def evaluate_check(check_name, fail_query):
        nonlocal checks_passed
        df = conn.execute(fail_query).df()
        if len(df) == 0:
            checks_passed += 1
            msg = f"[PASS] {check_name}"
            print(msg)
            log.append(msg)
        else:
            msg = f"[FAIL] {check_name} | {len(df)} failing rows. Example: {df.iloc[0].to_dict()}"
            print(msg)
            log.append(msg)

    # Check 1-4: daily_sales_by_category
    evaluate_check("1. No null order_date, region, or product_category", 
                   "SELECT * FROM gold_sales WHERE order_date IS NULL OR region IS NULL OR product_category IS NULL")
    
    evaluate_check("2. total_revenue > 0 for all rows", 
                   "SELECT * FROM gold_sales WHERE total_revenue <= 0") 
    
    evaluate_check("3. order_count > 0 for all rows", 
                   "SELECT * FROM gold_sales WHERE order_count <= 0")
    
    evaluate_check("4. No duplicate (order_date, region, product_category) combinations", 
                   "SELECT order_date, region, product_category, COUNT(*) as c FROM gold_sales GROUP BY 1,2,3 HAVING c > 1")

    # Check 5-8: returns_rate
    evaluate_check("5. No null order_date", 
                   "SELECT * FROM gold_returns WHERE order_date IS NULL")
    
    evaluate_check("6. returns_rate_pct between 0.0 and 100.0 inclusive", 
                   "SELECT * FROM gold_returns WHERE returns_rate_pct < 0 OR returns_rate_pct > 100")
    
    evaluate_check("7. returned_orders <= total_orders", 
                   "SELECT * FROM gold_returns WHERE returned_orders > total_orders")
    
    diff = conn.execute("SELECT MAX(order_date) - MIN(order_date) FROM gold_returns").fetchone()[0]
    if diff >= 30:
        checks_passed += 1
        msg = "[PASS] 8. order_date range spans at least 30 days"
        print(msg)
        log.append(msg)
    else:
        msg = f"[FAIL] 8. order_date range spans at least 30 days | Range is only {diff} days"
        print(msg)
        log.append(msg)

    summary = f"\nRESULT: {checks_passed}/{total_checks} checks passed."
    print(summary)
    log.append(summary)
    return log

# --- PHASE 1: Run on Clean Data ---
run_all_checks("CLEAN DATA")

# --- PHASE 2: Inject Bad Data & Verify it Fails ---
print("\n[!] Injecting bad row: Negative Revenue (-999.99)...")
conn.execute("INSERT INTO gold_sales VALUES (DATE '2024-01-01', 'North', 'Electronics', -999.99, 5);")
run_all_checks("DIRTY DATA (Should see 1 FAIL)")

# --- PHASE 3: Clean up & Generate Certificate ---
print("\n[!] Removing bad row...")
conn.execute("DELETE FROM gold_sales WHERE total_revenue = -999.99;")
final_log = run_all_checks("FINAL CLEAN RUN")

# Save the final passing log to our certificate file
with open('artefacts/700-wide/dq-certificate.md', 'w') as f:
    f.write("# Data Quality Certificate\n\n")
    f.write("\n".join(final_log))

print("\n✓ DQ Certificate saved to artefacts/700-wide/dq-certificate.md")

### --- CELL 7: SERVE THE DATA (DASHBOARD) --- ###

# 1. Install visualization libraries AND the missing notebook formatter
%pip install --quiet plotly streamlit nbformat

import pandas as pd
import plotly.express as px
import os

# 2. Load the Gold data (Using DuckDB to read it safely to avoid PyArrow version errors)
df_sales = conn.execute("SELECT * FROM read_parquet('gold/daily_sales_by_category.parquet')").df()
df_returns = conn.execute("SELECT * FROM read_parquet('gold/returns_rate.parquet')").df()

# --- INLINE NOTEBOOK VISUALIZATION ---
print("=== Sales Performance Dashboard (Inline Preview) ===\n")

# Metrics
total_rev = df_sales['total_revenue'].sum()
avg_return = df_returns['returns_rate_pct'].mean()
print(f"💰 Total Revenue: ${total_rev:,.2f}")
print(f"📉 Average Returns Rate: {avg_return:.2f}%\n")

# Chart 1: Revenue by Region (Grouped Bar)
sales_grouped = df_sales.groupby(['region', 'product_category'])['total_revenue'].sum().reset_index()
fig1 = px.bar(sales_grouped, x='region', y='total_revenue', color='product_category', barmode='group', 
              title="Total Revenue by Region & Category")
fig1.show()

# Chart 2: Returns Rate Over Time (Line)
df_returns = df_returns.sort_values('order_date')
fig2 = px.line(df_returns, x='order_date', y='returns_rate_pct', 
               title="Returns Rate Over Time (%)")
fig2.show()

print(f"Data last updated: {df_sales['order_date'].max()}")

# NOTE: One thing I would change about this dashboard is to format the Y-axis of the 
# first chart as actual currency ($) instead of raw numbers to make it easier to read!


# --- GENERATE STREAMLIT APP.PY ---
streamlit_code = """import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Dashboard", layout="wide")
st.title("Sales Performance Dashboard")

# Read data (Using absolute paths or assuming running from same dir)
df_sales = pd.read_parquet('gold/daily_sales_by_category.parquet')
df_returns = pd.read_parquet('gold/returns_rate.parquet')

# Metrics
total_rev = df_sales['total_revenue'].sum()
avg_return = df_returns['returns_rate_pct'].mean()
col1, col2 = st.columns(2)
col1.metric("Total Revenue", f"${total_rev:,.2f}")
col2.metric("Average Returns Rate", f"{avg_return:.2f}%")

st.divider()

# Chart 1
st.subheader("Total Revenue by Region")
sales_grouped = df_sales.groupby(['region', 'product_category'])['total_revenue'].sum().reset_index()
fig1 = px.bar(sales_grouped, x='region', y='total_revenue', color='product_category', barmode='group')
st.plotly_chart(fig1, use_container_width=True)

# Chart 2
st.subheader("Returns Rate Over Time")
df_returns = df_returns.sort_values('order_date')
fig2 = px.line(df_returns, x='order_date', y='returns_rate_pct')
st.plotly_chart(fig2, use_container_width=True)

st.caption(f"Data last updated: {df_sales['order_date'].max()}")
"""

# Save app.py to the workspace and the artefacts folder
with open('app.py', 'w') as f:
    f.write(streamlit_code)
    
with open('artefacts/700-wide/app.py', 'w') as f:
    f.write(streamlit_code)

print("\n✓ Streamlit app saved to app.py and artefacts/700-wide/app.py")

### --- CELL 8: AGENT-GENERATED PIPELINE (NEW DATASET) --- ###
import os
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import plotly.express as px

# 1. BRONZE: Generate Data
print("=== 1. BRONZE LAYER ===")
random.seed(100)
np.random.seed(100)
num_rows = 500

base_events = [f"EVT-{str(i).zfill(5)}" for i in range(1, num_rows - 9)]
dup_events = random.choices(base_events, k=10) # ~2% duplicates
event_ids = base_events + dup_events
random.shuffle(event_ids)

student_ids = np.random.randint(1000, 10000, size=num_rows)
categories = np.random.choice(['Data', 'Engineering', 'Design', 'Business', 'Security'], size=num_rows)

start_date = datetime(2024, 1, 1)
dates = [start_date + timedelta(days=random.randint(0, 364)) for _ in range(num_rows)]
date_formats = ["%Y-%m-%d", "%m/%d/%Y", "%b %d %Y"]
formatted_dates = [d.strftime(random.choice(date_formats)) for d in dates]

pcts = np.round(np.random.uniform(0.0, 100.0, size=num_rows), 1).astype(object)
null_indices = random.sample(range(num_rows), 20) # 4% nulls
for i in null_indices: pcts[i] = np.nan

times = np.random.randint(10, 481, size=num_rows)
statuses = np.random.choice(['completed', 'in_progress', 'dropped'], size=num_rows, p=[0.70, 0.20, 0.10])

df_events = pd.DataFrame({
    'event_id': event_ids, 'student_id': student_ids, 'event_date': formatted_dates,
    'course_category': categories, 'completion_pct': pcts, 
    'time_spent_minutes': times, 'status': statuses
})
df_events.to_csv('bronze/course_events_raw.csv', index=False)
print(f"Generated {len(df_events)} raw rows.")

# 2. SILVER: Clean Data
print("\n=== 2. SILVER LAYER ===")
silver_query = """
WITH parsed AS (
    SELECT 
        event_id, student_id,
        COALESCE(TRY_STRPTIME(event_date, '%Y-%m-%d'), TRY_STRPTIME(event_date, '%m/%d/%Y'), TRY_STRPTIME(event_date, '%b %d %Y'))::DATE AS event_date,
        course_category, TRY_CAST(completion_pct AS DOUBLE) AS completion_pct,
        time_spent_minutes, status
    FROM read_csv_auto('bronze/course_events_raw.csv', ALL_VARCHAR=1)
    WHERE completion_pct IS NOT NULL AND completion_pct != 'nan'
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY event_id ORDER BY student_id DESC) as rn FROM parsed
)
SELECT * EXCLUDE(rn) FROM ranked WHERE rn = 1
"""
conn.execute(f"COPY ({silver_query}) TO 'silver/course_events_clean.parquet' (FORMAT PARQUET);")
silver_count = conn.execute("SELECT COUNT(*) FROM read_parquet('silver/course_events_clean.parquet')").fetchone()[0]
print(f"Cleaned data: {silver_count} rows (removed nulls & duplicates).")

# 3. GOLD: Business Metrics (WITH A DELIBERATE MISTAKE!)
print("\n=== 3. GOLD LAYER ===")
conn.execute("""
CREATE OR REPLACE TABLE gold_completions AS 
SELECT event_date, course_category, AVG(completion_pct) as avg_completion_pct, COUNT(event_id) as completion_count
FROM read_parquet('silver/course_events_clean.parquet')
WHERE status = 'completed'
GROUP BY 1, 2;
""")

# MISTAKE: Look closely at the denominator in the dropout_rate_pct calculation!
conn.execute("""
CREATE OR REPLACE TABLE gold_dropout AS 
SELECT 
    event_date,
    COUNT(event_id) AS total_enrollments,
    COUNT(CASE WHEN status = 'dropped' THEN event_id END) AS dropped_count,
    ROUND(COUNT(CASE WHEN status = 'dropped' THEN event_id END) * 100.0 / 
         NULLIF(COUNT(event_id), 0), 2) AS dropout_rate_pct
FROM read_parquet('silver/course_events_clean.parquet')
GROUP BY 1;
""")
print("Gold tables created.")

# 4. DQ CHECKS
print("\n=== 4. DQ CHECKS ===")
dq_fails = 0
if conn.execute("SELECT COUNT(*) FROM gold_completions WHERE event_date IS NULL").fetchone()[0] > 0: dq_fails += 1
if conn.execute("SELECT COUNT(*) FROM gold_completions WHERE avg_completion_pct < 0 OR avg_completion_pct > 100").fetchone()[0] > 0: dq_fails += 1
if conn.execute("SELECT COUNT(*) FROM gold_dropout WHERE dropout_rate_pct < 0 OR dropout_rate_pct > 100").fetchone()[0] > 0: dq_fails += 1
print(f"DQ Checks complete. Failures found: {dq_fails}")

# 5. SERVE
print("\n=== 5. SERVE (CHARTS) ===")
df_drop = conn.execute("SELECT * FROM gold_dropout ORDER BY event_date").df()
fig_drop = px.line(df_drop, x='event_date', y='dropout_rate_pct', title="Dropout Rate Over Time (%)")
fig_drop.show()

658.85s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
--- Query Results ---
   id                   message                 created_at
0   1  Hello from Bronze layer! 2026-09-27 00:32:23.574065
1   2  Hello from Silver layer! 2026-09-27 00:32:23.574065
2   3    Hello from Gold layer! 2026-09-27 00:32:23.574065
--------------------
Environment ready ✓

--- Bronze Data Generation Complete ---
Total rows: 500
Null amount count: 25
Duplicate order_id count: 15
Unique date formats used: ['%Y-%m-%d', '%d/%m/%Y', '%b %d %Y']

--- DuckDB Profiling Results ---
|   total_rows |   null_amount_count |   duplicate_order_ids |   min_amount |   max_amount | distinct_statuses                  |
|-------------:|--------------------:|----------------------:|-------------:|-------------:|:-----------------------------------|
|          500 |                  25 |  

665.09s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
=== Sales Performance Dashboard (Inline Preview) ===

💰 Total Revenue: $89,050.49
📉 Average Returns Rate: 14.25%



Data last updated: 2024-12-30 00:00:00

✓ Streamlit app saved to app.py and artefacts/700-wide/app.py
=== 1. BRONZE LAYER ===
Generated 500 raw rows.

=== 2. SILVER LAYER ===
Cleaned data: 470 rows (removed nulls & duplicates).

=== 3. GOLD LAYER ===
Gold tables created.

=== 4. DQ CHECKS ===
DQ Checks complete. Failures found: 0

=== 5. SERVE (CHARTS) ===
